In [ ]:
from CodeSensor import *
from run import load_model, load_tokenizer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

In [ ]:
df_cpp = pd.read_parquet('tests/cpp_tests_10k.parquet')
df_python = pd.read_parquet('tests/python_tests_10k.parquet')

In [ ]:
from torch.cuda import is_available
device = 'cuda:0' if is_available() else 'cpu'

tokenizer = load_tokenizer()
model_cpp = load_model('weights/cpp_old.pth', device)
model_py = load_model('weights/py_v2.pth', device)

sensor_cpp = CodeSensor(tokenizer, model_cpp, device)
sensor_py = CodeSensor(tokenizer, model_py, device)

### Инструкция по использованию ноутбука
1. Запустите ячейку с импортами и загрузкой зависимостей.
2. Запустите ячейку, которая загружает датасеты `tests/cpp_tests_10k.parquet` и `tests/python_tests_10k.parquet`.
3. Запустите ячейку, где инициализируются модель и сенсоры (`sensor_cpp`, `sensor_py`).
4. Используйте блок "Пример тестирования модели для всего датасета" для оценки качества на выборке.
5. В блоке "Тест произвольного кода" подставьте свой код в переменную `custom_code`, выберите `custom_lang = 'py'` или `custom_lang = 'cpp'`, и запустите ячейку.
6. Результат анализа выводится в ячейку, а HTML-отчет сохраняется в файл `custom_test_result.html`.

### Что проверяется
- `verdict` — итоговый статус: 0 = человек, 1 = подозрительно, 2 = опасно
- `confidence` — максимальная вероятность подозрительного фрагмента
- `suspicious tokens` — количество подозрительных токенов
- `threshold` — порог оценки для подсветки

In [ ]:
df_cpp['verdict'] = df_cpp['code'].apply(lambda code : sensor_cpp.analyze(code).verdict)

df_cpp['pred_label'] = df_cpp['verdict'].apply(lambda v: 1 if v >= 1 else 0)
df_cpp['true_label'] = df_cpp['label'].apply(lambda x: 0 if x == 'HG' else 1)

df_cpp['fp_rate'] = df_cpp.apply(lambda row : int(row['pred_label'] == 1 and row['true_label'] == 0), axis=1)
df_cpp['fn_rate'] = df_cpp.apply(lambda row : int(row['pred_label'] == 0 and row['true_label'] == 1), axis=1)

fp = int(np.sum(df_cpp['fp_rate']))
fn = int(np.sum(df_cpp['fn_rate']))

tn = int(((df_cpp['pred_label'] == 0) & (df_cpp['true_label'] == 0)).sum())
tp = int(((df_cpp['pred_label'] == 1) & (df_cpp['true_label'] == 1)).sum())

accuracy = accuracy_score(df_cpp['true_label'], df_cpp['pred_label'])
roc_auc = roc_auc_score(df_cpp['true_label'], df_cpp['verdict'].apply(lambda v: v / 2.0))

print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"Accuracy: {accuracy:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"Confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

### Тест произвольного кода
Введите фрагмент Python или C/C++ кода и запустите модель для проверки.

In [ ]:
custom_code = """
# Пример Python-кода
for i in range(5):
    print(i)
"""

custom_lang = 'py'  # 'py' для Python, 'cpp' для C/C++

sensor = sensor_py if custom_lang == 'py' else sensor_cpp
result = sensor.analyze(custom_code)

print('verdict:', result.verdict)
print('confidence:', f"{result.confidence:.4f}")
print('suspicious tokens:', result.suspicious_count)
print('threshold:', f"{result.threshold:.2f}")

from HtmlVisualizer import HtmlVisualizer
html_output = HtmlVisualizer.render(custom_code, result)

# Если хотите, можно сохранить результат как HTML-файл:
with open('custom_test_result.html', 'w', encoding='utf-8') as f:
    f.write(html_output)

print('HTML result saved to custom_test_result.html')